# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!git clone https://github.com/shrisha-rao/schemapilot.git

fatal: destination path 'schemapilot' already exists and is not an empty directory.


In [3]:
%cd schemapilot

/content/schemapilot


In [4]:
!git pull origin main

From https://github.com/shrisha-rao/schemapilot
 * branch            main       -> FETCH_HEAD
Already up to date.


In [5]:
!ls

api   docker-compose.yml  Dockerfile~  model	  README.md	   results
data  Dockerfile	  eval	       notebooks  requirments.txt  src


In [6]:
!pip install --upgrade pip
!pip install -r requirments.txt

# 1. Load Model & Tokenizer

In [7]:
#import unsloth
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# 2. Apply LoRA


In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Higher rank helps with strict formatting
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


In [10]:
!pwd

/content/schemapilot


# 3. Load & Format Data

In [25]:
def formatting_prompts_func(examples):
    # Extract the exact columns from the Salesforce dataset
    queries = examples["query"]
    tools   = examples["tools"]
    answers = examples["answers"]

    texts = []

    for query, tool, answer in zip(queries, tools, answers):
        # 1. Define the System Prompt to enforce the "Zero-Talk" Agent persona
        system_prompt = (
            "You are a precision routing agent. You must output a valid JSON array "
            "of function calls based ONLY on the provided tools. No conversational filler."
        )

        # 2. Stitch everything together into the Prompt Template
        text = (
            f"### System:\n{system_prompt}\n\n"
            f"### Available Tools:\n{tool}\n\n"
            f"### User Query:\n{query}\n\n"
            f"### Agent Response:\n{answer}{tokenizer.eos_token}"
        )

        texts.append(text)

    return texts

In [26]:
## Pipeline test dataset
# dataset = load_dataset("json",
#                         data_files="data/training_data.jsonl",
#                         split="train")

In [28]:
!pip install -q huggingface_hub
from huggingface_hub import login

In [31]:
import os
hf_token = os.getenv('HF')

In [33]:
login()

In [35]:
# FULL dataset
from datasets import load_dataset

# Load the first 10000 rows from the Salesforce dataset
dataset = load_dataset("Salesforce/xlam-function-calling-60k",
                       split="train[:10000]")

xlam_function_calling_60k.json:   0%|          | 0.00/96.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

# 4. Training Arguments

In [64]:

# Run the Trainer again
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    formatting_func = formatting_prompts_func,
    max_seq_length = 2048,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        # "Effective Batch Size" = 16
        gradient_accumulation_steps = 16, # Adding this to keep memory stable
        warmup_steps = 5,
        max_steps = 350,
        learning_rate = 2e-4,
        fp16 = True, # not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        report_to = "wandb"
    ),
)

# Clean any leftover GPU cache before starting
torch.cuda.empty_cache()

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 350
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 25,165,824 of 3,846,245,376 (0.65% trained)


Step,Training Loss
1,0.438800
2,0.568300
3,0.551800
4,0.560600
5,0.557600
6,0.527700
7,0.522300
8,0.442700
9,0.651900
10,0.505700


train/epoch,▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▁▁▁▂▂▃▃▃▃▄▄▄▄▅▅▆▆▆▆▇▇▇▇█
train/global_step,▁▁▂▂▂▃▃▃▃▃▅▅▅▅▅▁▁▁▁▂▂▃▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
train/grad_norm,▃▄▃▄▃▃▄▄▄▅▃▁▂▁▁▁▃▂▂▂▂▂▃▂█▃▂▂▃▃▂▃▃▂▂▂▂▂▃▄
train/learning_rate,▂▃▅▅▄▄▄▄▄▄▃▃▃▃▃▂██▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▂▂▁
train/loss,█▇▅▅▅▅▄▂▃▇█▆▃█▅▅▅▃▆▄▅▄▃▃▄▄▅▄▃▄▁▃▃▄▄▃▂▃▃▃
total_flos,6.002223119659008e+16
train/epoch,0.56
train/global_step,350
train/grad_norm,0.20682
train/learning_rate,0.0
train/loss,0.4508


TrainOutput(global_step=350, training_loss=0.4859634716170175, metrics={'train_runtime': 3500.1857, 'train_samples_per_second': 1.6, 'train_steps_per_second': 0.1, 'total_flos': 6.002223119659008e+16, 'train_loss': 0.4859634716170175, 'epoch': 0.56})

## Save LoRA adapted model

In [65]:
from pathlib import Path

# Choose the checkpoint you want (e.g., the final one)
ckpt_dir = Path("outputs/checkpoint-350")   # replace <last_step>

# Save a self‑contained model that includes the LoRA adapters
model.save_pretrained("fine_tuned_model")
tokenizer.save_pretrained("fine_tuned_model")

('fine_tuned_model/tokenizer_config.json',
 'fine_tuned_model/special_tokens_map.json',
 'fine_tuned_model/chat_template.jinja',
 'fine_tuned_model/tokenizer.model',
 'fine_tuned_model/added_tokens.json',
 'fine_tuned_model/tokenizer.json')

# Eval

In [66]:
# Load fine tuned model
fineTunedModel, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "fine_tuned_model",   # path to the folder
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "auto",
)

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [67]:
from src.eval import evaluate_model
eval_stats = evaluate_model(fineTunedModel,
                            tokenizer,
                            ["Book a room", "Weather in London"])

In [68]:
eval_stats

{'valid_json': 0,
 'correct_tool': 0,
 'latency': [3.8943283557891846, 2.774492025375366]}

In [69]:
user_query = "Book a room"
system_prompt = ("You are a precision routing agent. You must output a valid JSON array "
            "of function calls based ONLY on the provided tools. No conversational filler.")
prompt = (f"### System:\n{system_prompt}\n\n"
          f"### User Query:\n{user_query}\n\n"
          f"### Agent Response:\n")
tmp_inputs = tokenizer(prompt,
                           return_tensors="pt").to("cuda")
tmp_outputs = fineTunedModel.generate(**tmp_inputs,
                                      max_new_tokens=128,
                                      temperature=0.1)
#                                      stopping_criteria=tokenizer.eos_token)
tmp_prediction = tokenizer.decode(tmp_outputs[0],
                                  skip_special_tokens=True)


In [70]:
print(tmp_prediction)

### System:
You are a precision routing agent. You must output a valid JSON array of function calls based ONLY on the provided tools. No conversational filler.

### User Query:
Book a room

### Agent Response:
[{"name": "book_room", "arguments": {"room_type": "single", "nightly_rate": 120, "checkin": "2023-04-15", "checkout": "2023-04-17"}}]


In [3]:
tmp_prediction="""
### System:
You are a precision routing agent. You must output a valid JSON array of function calls based ONLY on the provided tools. No conversational filler.

### User Query:
Book a room

### Agent Response:
[{"name": "book_room", "arguments": {"room_type": "single", "nightly_rate": 120, "checkin": "2023-04-15", "checkout": "2023-04-17"}}]
"""

tmp_prediction.split('### Agent Response:\n')[-1]

'[{"name": "book_room", "arguments": {"room_type": "single", "nightly_rate": 120, "checkin": "2023-04-15", "checkout": "2023-04-17"}}]\n'

In [4]:
import json
json.loads(tmp_prediction.split('### Agent Response:\n')[-1])

[{'name': 'book_room',
  'arguments': {'room_type': 'single',
   'nightly_rate': 120,
   'checkin': '2023-04-15',
   'checkout': '2023-04-17'}}]

In [ ]:
tmp_prediction